In [1]:
!pip install flash-attn sympy math_verify pylatexenc


  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'pylatexenc' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pylatexenc'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=088dc091d63a73dad8035c35cdfcc572eefc97315deefcd6e60817ae91e0457b
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [math_verify] [latex2sympy2_extended]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --u

In [2]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='Qwen/Qwen2.5-Math-1.5B')

INFO 08-26 22:15:38 [__init__.py:244] Automatically detected platform rocm.


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

INFO 08-26 22:15:56 [config.py:853] This model supports multiple tasks: {'classify', 'embed', 'score', 'generate', 'reward'}. Defaulting to 'generate'.


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO 08-26 22:15:56 [config.py:1467] Using max model len 4096
INFO 08-26 22:16:04 [config.py:2267] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-26 22:16:04 [config.py:4566] full_cuda_graph is not supported with cascade attention. Disabling cascade attention.


...

WARNING 08-26 22:16:05 [utils.py:2613] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 08-26 22:16:08 [__init__.py:244] Automatically detected platform rocm.
INFO 08-26 22:16:17 [core.py:459] Waiting for init message from front-end.
INFO 08-26 22:16:17 [core.py:69] Initializing a V1 LLM engine (v0.9.2.dev364+gb432b7a28) with config: model='Qwen/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  d

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.05s/it]



INFO 08-26 22:16:21 [default_loader.py:272] Loading weights took 1.13 seconds
INFO 08-26 22:16:21 [gpu_model_runner.py:1782] Model loading took 3.0352 GiB and 3.966783 seconds
INFO 08-26 22:16:26 [backends.py:509] Using cache directory: /root/.cache/vllm/torch_compile_cache/d38fb0b184/rank_0_0/backbone for vLLM's torch.compile
INFO 08-26 22:16:26 [backends.py:520] Dynamo bytecode transform time: 4.54 s
INFO 08-26 22:16:41 [backends.py:181] Cache the graph of shape None for later use
INFO 08-26 22:16:41 [backends.py:193] Compiling a graph for general shape takes 13.00 s
INFO 08-26 22:16:43 [monitor.py:34] torch.compile takes 17.54 s in total
INFO 08-26 22:16:57 [gpu_worker.py:232] Available KV cache memory: 163.11 GiB
INFO 08-26 22:16:57 [kv_cache_utils.py:716] GPU KV cache size: 6,108,336 tokens
INFO 08-26 22:16:57 [kv_cache_utils.py:720] Maximum concurrency for 4,096 tokens per request: 1491.29x
INFO 08-26 22:16:57 [rocm.py:224] Using Triton Attention backend on V1 engine.


Capturing CUDA graphs: 100%|██████████| 67/67 [00:13<00:00,  5.13it/s]


INFO 08-26 22:17:10 [gpu_model_runner.py:2306] Graph capturing finished in 13 secs, took 0.27 GiB
INFO 08-26 22:17:10 [core.py:172] init engine (profile, create kv cache, warmup model) took 48.32 seconds


In [3]:
question = "Simplify $(3-i)(6+2i)$." 

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=0.5, top_p=0.5, max_tokens=1024, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0% 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Simplify $(3-i)(6+2i)$.\nAssistant: <think>'
Generated text: ' To simplify the expression $(3-i)(6+2i)$, we can use the distributive property (also known as the FOIL method for binomials). We multiply each term in the first binomial by each term in the second binomial. This gives us $3 \\cdot 6 + 3 \\cdot 2i - i \\cdot 6 - i \\cdot 2i$. Simplifying this, we get $18 + 6i - 6i - 2i^2$. Since $i^2 = -1$, the expression becomes $18 - 2(-1) = 18 + 2 = 20$. <answer> $20$ </answer>'


In [4]:
from drgrpo_grader import r1_zero_reward_fn


/shared-docker/drgrpo_grader.py:45: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$", answer)
/shared-docker/drgrpo_grader.py:320: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")
/shared-docker/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/shared-docker/drgrpo_grader.py:673: SyntaxWarning: invalid escape sequence '\^'
  BAD_REGEXES = ["\^[0-9]+\^", "\^[0-9][0-9]+"]
/shared-docker/drgrpo_grader.py:753: SyntaxWarning: invalid escape sequence '\d'
  p1 = re.compile("(\d)(,)(\d\d\d)($|\D)")
/shared-docker/drgrpo_grader.py:768: SyntaxWarning: invalid escape sequence '\{'
  m = re.search("^\\\\text\{(?P<text>.+?)\}$", expr)
/shared-docker/drgrpo_grader.py:801: SyntaxWarning: invalid escape sequence '\^'
  expr = re.sub(f"{unit}(es)?(s)? *(\^[0-9]+)?", "", expr)
/shared-docker/drgrpo_grader.py:802: SyntaxWarning: invalid escape sequence '\^'
  expr = re

In [5]:
r1_zero_reward_fn(generated_text, '20')


{'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}

In [6]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  # format_reward = 0.
  # answer_reward = 0.
  # reward = 0.
  output_rewards = []
      
  for index, output in enumerate(outputs):
    prompt = output.prompt
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    output_rewards.append(rewards)
    # format_reward += rewards['format_reward']
    # answer_reward += rewards['answer_reward']
    # reward += rewards['reward']
    # if rewards['reward'] == 1.0:
    #     print(f"Prompt: {prompt!r}")
    #     print(f"Generated text: {generated_text!r}")
    #     print(f"Answer: {answers[index]}")
    #     print(f"Reward: {rewards}")
    #     print('*****************')

  return output_rewards

In [7]:
import pandas as pd
from tqdm import tqdm
import random 

df = pd.read_parquet("math_12k.parquet")

total_format_reward = 0.
total_answer_reward = 0.
total_reward = 0.
prompts = []
answers = []

for index, row in tqdm(df.iterrows()):
    question = row['problem']
    answer = row['solution']
    prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""
    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 240 == 0:
        # Initialize batch counters
        batch_format_reward = 0
        batch_answer_reward = 0
        batch_total_reward = 0
        
        shots = []
        for i in range(4):
            shots.append(evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params))

        for r0, r1, r2, r3 in zip(shots[0], shots[1], shots[2], shots[3]):
            # if random.randint(1,20) == 20:
            #     print(f"Rewards: r0={r0}, r1={r1}, r2={r2}, r3={r3}")
            
            if (r0['format_reward'] == 1.0 or r1['format_reward'] == 1.0 or r2['format_reward'] == 1.0 or r3['format_reward'] == 1.0):
                batch_format_reward += 1
            if (r0['answer_reward'] == 1.0 or r1['answer_reward'] == 1.0 or r2['answer_reward'] == 1.0 or r3['answer_reward'] == 1.0):
                batch_answer_reward += 1
            if (r0['reward'] == 1.0 or r1['reward'] == 1.0 or r2['reward'] == 1.0 or r3['reward'] == 1.0):
                batch_total_reward += 1
        
        # Add batch results to totals
        total_format_reward += batch_format_reward
        total_answer_reward += batch_answer_reward
        total_reward += batch_total_reward
        
        print(f"Batch reward: {batch_total_reward}")
        print(f"Total reward so far: {total_reward}")
        prompts = []
        answers = []

print(f"Total format reward: {total_format_reward}")
print(f"Total answer reward: {total_answer_reward}")
print(f"Total reward: {total_reward}")

0it [00:00, ?it/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/240 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/240 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0% 0/240 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/240 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

12500it [30:45,  6.77it/s]

Batch reward: 95
Total reward so far: 4007.0
Total format reward: 8130.0
Total answer reward: 4007.0
Total reward: 4007.0
